# Clinical Exploratory Data Analysis & Statistical Profiling
## UCI Heart Disease Risk Scoring: Clinical Interpretability vs Accuracy

> **Clinical Context**: In a cardiovascular triage or referral clinic, identifying coronary artery disease (CAD >= 50% luminal stenosis) early prevents catastrophic ischemic events (e.g. myocardial infarction, ventricular arrhythmia, cardiogenic death). A missed case (false negative) carries life-threatening clinical risk, whereas a false positive merely leads to a secondary non-invasive stress test or cardiologist evaluation.

### Dataset Overview
- **Source**: Cleveland Clinic Foundation (UCI ID: 45)
- **Patients**: 303
- **Features**: 13 (5 continuous, 8 categorical)
- **Target**: Binary CAD status (`0` = healthy/no CAD, `1` = CAD present)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Ensure project modules are accessible
sys.path.insert(0, str(Path.cwd().parent))
from src.data import load_raw_data, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET_COL

X, y = load_raw_data()
df = pd.concat([X, y], axis=1)
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Clinical Interpretation of Continuous Features

| Feature | Unit | Clinical Rationale & Pathophysiology | Expected Relationship with CAD |
| :--- | :--- | :--- | :--- |
| **age** | Years | Cumulative arterial wear, endothelial senescence, and atherosclerotic plaque accumulation. | Strong positive association (older -> higher risk). |
| **trestbps** | mm Hg | Resting systolic blood pressure. Systemic hypertension causes endothelial micro-injury, shear stress, and promotes vascular remodeling. | Moderate positive association. |
| **chol** | mg/dl | Serum cholesterol. Elevated low-density lipoprotein (LDL) particles infiltrate the intima and form oxidized atheromatous cores. | Moderate positive association. |
| **thalach** | bpm | Peak exercise heart rate achieved. Reflects autonomic reserve and left ventricular functional reserve; **chronotropic incompetence** indicates severe ischemia. | Strong negative association (lower peak HR -> severe disease). |
| **oldpeak** | mm | Exercise-induced ST-segment depression relative to baseline rest. Classic diagnostic hallmark of subendocardial myocardial ischemia. | Strong positive association (deeper depression -> multi-vessel disease). |

In [ ]:
# Mann-Whitney U Non-parametric Hypothesis Testing
print("=== Mann-Whitney U Tests: Continuous Risk Factors ===")
for feat in NUMERICAL_FEATURES:
    group0 = df[df[TARGET_COL] == 0][feat].dropna()
    group1 = df[df[TARGET_COL] == 1][feat].dropna()
    stat, p = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{feat:10s} | Healthy: {group0.median():.1f} (IQR {group0.quantile(0.75)-group0.quantile(0.25):.1f}) vs CAD: {group1.median():.1f} (IQR {group1.quantile(0.75)-group1.quantile(0.25):.1f}) | p-value: {p:.2e}")

## 2. Clinical Interpretation of Categorical Features

- **`cp` (Chest Pain Type)**: Anginal pain is classified into:
  1. *Typical angina*: Substernal, provoked by exertion, relieved by rest/NTG.
  2. *Atypical angina*: 2 of 3 features.
  3. *Non-anginal pain*: 1 feature.
  4. *Asymptomatic*: In referral centers like Cleveland, patients with atypical or 'asymptomatic' presentation referred for cardiac catheterization typically had severe non-invasive findings, hence showing high CAD rates.
- **`ca` (Major Vessels Colored by Fluoroscopy, 0-3)**: Direct anatomic visualization of calcified, obstructed coronary vessels. 1, 2, or 3 vessels indicate progressive multi-vessel coronary artery disease.
- **`thal` (Thallium Scintigraphy)**: Myocardial perfusion scan. `3` = normal perfusion; `6` = fixed defect (previous myocardial necrosis/scar); `7` = reversible defect (ischemia in viable myocardium at acute risk of infarction).
- **`exang` (Exercise-induced Angina)**: Mismatch between coronary blood flow reserve and myocardial oxygen demand under treadmill stress.

In [ ]:
# Chi-Square Contingency Tests
print("=== Chi-Square Contingency Tests: Categorical Risk Factors ===")
for col in ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']:
    table = pd.crosstab(df[col], df[TARGET_COL])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    print(f"{col:8s} | Chi2: {chi2:7.2f} | dof: {dof} | p-value: {p:.2e}")

## 3. Missing Data & Clinical Imputation Strategy
- `ca`: 4 missing values (~1.3%)
- `thal`: 2 missing values (~0.7%)
- Missing data occurs completely at random (MCAR) or due to imaging protocol exclusions. Because the missing proportion is negligible (<2%), we use median/mode imputation within a leak-free scikit-learn Pipeline (fit strictly on training folds).